In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ---- Device (Mac: MPS if available, else CPU) ----
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

# ---- Data ----
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False)

# ---- Model (simple MLP) ----
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(model)

# ---- Training setup ----
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

# ---- Training loop ----
epochs = 2
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0

    for step, (x, y) in enumerate(train_loader, start=1):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if step % 100 == 0:
            avg_loss = running_loss / 100
            running_loss = 0.0
            print(f"Epoch {epoch} Step {step}: loss={avg_loss:.4f}")

    test_acc = accuracy(model, test_loader)
    print(f"Epoch {epoch} finished. Test accuracy: {test_acc:.4%}")


Device: mps


100%|██████████| 9.91M/9.91M [00:00<00:00, 29.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.01MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.8MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.08MB/s]


MLP(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=10, bias=True)
  )
)
Epoch 1 Step 100: loss=0.5141
Epoch 1 Step 200: loss=0.2487
Epoch 1 Step 300: loss=0.2109
Epoch 1 Step 400: loss=0.1753
Epoch 1 finished. Test accuracy: 96.0900%
Epoch 2 Step 100: loss=0.1170
Epoch 2 Step 200: loss=0.1153
Epoch 2 Step 300: loss=0.1082
Epoch 2 Step 400: loss=0.1001
Epoch 2 finished. Test accuracy: 97.2300%
